# Silver Layer — Orders
## SalesFlow Data Lakehouse | Phase 4: Curated Layer

Reads `salesflow_dev.bronze.orders`, applies date validation and business logic,
and writes the curated result to `salesflow_dev.silver.orders`.

**Transformations applied:**
| Step | Transformation |
|---|---|
| 1 | Remove duplicates by `OrderID` |
| 2 | Validate `OrderDate` is not in the future |
| 3 | Add `is_shipped` flag |
| 4 | Calculate `days_to_ship` |
| 5 | Add `data_quality_status` flag |
| 6 | Add `processing_timestamp` |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Bronze

In [0]:
from pyspark.sql.functions import (
    current_timestamp, current_date, when, col,
    datediff, to_date
)

# Read orders table from Bronze layer
df = spark.table("salesflow_dev.bronze.orders")

print(f"Records read from Bronze: {df.count()}")
display(df.limit(5))

## 2. Remove Duplicates
Deduplicate by primary key `OrderID`.

In [0]:
# Drop duplicate OrderIDs — primary key must be unique in Silver
df = df.dropDuplicates(["OrderID"])

print(f"Records after deduplication: {df.count()}")

## 3. Cast Date Columns
Ensure date columns are properly typed before applying any date logic.

In [0]:
# Cast date strings to DateType for date operations
df = df \
    .withColumn("OrderDate",    to_date(col("OrderDate"))) \
    .withColumn("ShippedDate",  to_date(col("ShippedDate"))) \
    .withColumn("RequiredDate", to_date(col("RequiredDate")))

## 4. Validate Dates
`OrderDate` cannot be in the future — flag invalid records rather than dropping them.

In [0]:
# Cell 8's to_date() without format fails - reload from bronze and parse correctly
from pyspark.sql.functions import substring

# Reload from bronze with deduplication (Cell 6's logic)
df = spark.table("salesflow_dev.bronze.orders").dropDuplicates(["OrderID"])

# Parse date columns with correct format
df = df \
    .withColumn("OrderDate",    to_date(substring(col("OrderDate"), 1, 10), "yyyy/MM/dd")) \
    .withColumn("ShippedDate",  to_date(substring(col("ShippedDate"), 1, 10), "yyyy/MM/dd")) \
    .withColumn("RequiredDate", to_date(substring(col("RequiredDate"), 1, 10), "yyyy/MM/dd"))

# Add date validation flag: INVALID if OrderDate is null or in the future
df = df.withColumn(
    "date_valid",
    when(
        col("OrderDate").isNull() | (col("OrderDate") > current_date()),
        False
    ).otherwise(True)
)

# Preview how many records failed date validation
print("Date validation distribution:")
display(df.groupBy("date_valid").count())

## 5. Add Business Logic Columns

### `is_shipped`
`TRUE` if `ShippedDate` is not null — order has left the warehouse.

### `days_to_ship`
Number of days between `OrderDate` and `ShippedDate`.  
`NULL` if the order has not been shipped yet.

In [0]:
# 5.1 is_shipped: TRUE if ShippedDate is populated
df = df.withColumn(
    "is_shipped",
    when(col("ShippedDate").isNotNull(), True).otherwise(False)
)

# 5.2 days_to_ship: only calculated for shipped orders
df = df.withColumn(
    "days_to_ship",
    when(
        col("is_shipped"),
        datediff(col("ShippedDate"), col("OrderDate"))
    ).otherwise(None)  # null for unshipped orders
)

## 6. Add Quality Flag
A record is `INVALID` if any of the following is true:
- `OrderID` is null
- `CustomerID` is null
- `OrderDate` is null or in the future
- `days_to_ship` is negative (shipped before order was placed)

In [0]:
# Custom quality condition combining null checks and business rules
df = df.withColumn(
    "data_quality_status",
    when(
        col("OrderID").isNull() |
        col("CustomerID").isNull() |
        (~col("date_valid")) |
        (col("days_to_ship") < 0),
        "INVALID"
    ).otherwise("VALID")
)

# Drop helper column — no longer needed after quality flag is set
df = df.drop("date_valid")

## 7. Add Processing Timestamp

In [0]:
# Capture when this record was processed in the Silver layer
df = df.withColumn("processing_timestamp", current_timestamp())

## 8. Save as Delta Table

In [0]:
# Write to Silver layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.silver.orders")

print("Table saved: salesflow_dev.silver.orders")

## 9. Validation

In [0]:
silver_orders = spark.table("salesflow_dev.silver.orders")

# Record count
print(f"Total records: {silver_orders.count()}")

# Quality flag distribution
print("\nQuality flag distribution:")
display(silver_orders.groupBy("data_quality_status").count())

# Shipping status distribution
print("\nShipping status distribution:")
display(silver_orders.groupBy("is_shipped").count())

# days_to_ship stats — useful to catch outliers
print("\nDays to ship statistics:")
display(silver_orders.select("days_to_ship").summary())

# Schema
print("\nSchema:")
silver_orders.printSchema()

# Sample
print("\nFirst 5 rows:")
display(silver_orders.limit(5))